In [3]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Doutorado/Disciplinas/4 Trimestre/GA036 - Introdução a ML/data_radar_preprocessing

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Doutorado/Disciplinas/4 Trimestre/GA036 - Introdução a ML/data_radar_preprocessing


In [4]:
import tensorflow as tf

@tf.function
def train_step(x, y_true, model, loss_fn):
  train_dataset = tf.data.Dataset.from_tensor_slices((x, y_true))
  for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
    with tf.GradientTape() as tape:
        y_pred = model(x_batch_train, training=True)

        # Create nan mask
        nan_mask = tf.math.is_nan(y_batch_train)
        nan_mask = tf.squeeze(nan_mask) # Remove extra dimensions if any

        y_pred_masked = tf.boolean_mask(y_pred, nan_mask)
        y_true_masked = tf.boolean_mask(y_batch_train, nan_mask)

        loss = loss_fn(y_true_masked, y_pred_masked)

    grads = tape.gradient(loss, model.trainable_variables)
    model.optimizer.apply_gradients(zip(grads, model.trainable_variables))

  return loss

def custom_fit(model, x, y_true, num_epochs=100, loss_fn=tf.keras.losses.MeanSquaredError()):
    for i in range(num_epochs):
        loss = train_step(x, y_true, model, loss_fn)
        if i % 10 == 0:
            print(f"Epoch {i}, Loss: {loss.numpy()}")
    return model

In [5]:
import os
import numpy as np
import xarray as xr
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold, ParameterGrid

from models.conv_lstm import build_model as ConvLSTMArch
from models.dlinear import build_dlinear as DLinearArch

cfg = {
    "data_path": "output_data/train_data.nc",
    "target_var": "dbzh",
    "architecture": "ConvLSTM",  # "ConvLSTM" or "DLinear"
    "seq_len": 10,
    "pred_len": 1,
    "epochs": 1,
    "batch_size": 32,
    "n_splits": 5,
    "output_path": "outputs/",
    "exp_name": "test_run_no_fill"
}

def prepare_data(arr, seq_len, pred_len=1):
    timestamps = arr.shape[0]
    data_points = 1 # e.g., 1 variable/feature
    altitude = arr.shape[-1]
    latitude = arr.shape[1]
    longitude = arr.shape[2]

    arr = arr.reshape(timestamps, data_points, altitude, latitude, longitude)

    if cfg.get("architecture") == "DLinear":
        # Flatten spatial dims -> cada ponto espacial vira um feature
        features = data_points * altitude * latitude * longitude
        arr = arr.reshape(timestamps, features)  # (timestamps, features)

    X, y = [], []

    for i in range(timestamps - seq_len - pred_len + 1):
        seq_x = arr[i:i+seq_len]
        # seq_y = arr[i+seq_len:i+seq_len+pred_len]
        seq_y = arr[i+seq_len]

        if cfg.get("architecture") != "DLinear":
            seq_x = np.transpose(seq_x, (0, 2, 3, 4, 1))  # (seq_len, alt, lat, lon, channels)
            seq_y = np.transpose(seq_y, (1, 2, 3, 0))    # (alt, lat, lon, channels)

        X.append(seq_x)
        y.append(seq_y)

    X = np.array(X)
    y = np.array(y)

    return X, y

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def build_model(params=None):
    model = None
    if cfg.get("architecture") == "ConvLSTM":
        model = ConvLSTMArch(input_shape=X_train.shape[1:], filters=params["filters"])
    elif cfg.get("architecture") == "DLinear":
        model = DLinearArch(seq_len=cfg["seq_len"], features=features, pred_len=cfg["pred_len"])
    return model

In [6]:
ds = xr.open_dataset(cfg.get("data_path"))
arr = ds[cfg.get("target_var")].values

param_grid = {
    "filters": [[16,8], [32,16]],
    "batch_size": [8, 16],
    "epochs": [50, 100]
}

seed = 42
grid = list(ParameterGrid(param_grid))
print(f"Total combinations: {len(grid)}")
kf = KFold(n_splits=cfg.get("n_splits", 3), shuffle=True, random_state=seed)

timestamps = arr.shape[0]
data_points = 1
altitude = arr.shape[-1]
latitude = arr.shape[1]
longitude = arr.shape[2]
seq_len = cfg.get("seq_len")

print(f"Selected model: {cfg.get('architecture')}")
print(f"Params: cfg={cfg}")

X, y = prepare_data(arr, cfg["seq_len"], cfg["pred_len"])

print("X shape:", X.shape)
print("y shape:", y.shape)

results = []
model_id = 0

params = grid[2]

Total combinations: 8
Selected model: ConvLSTM
Params: cfg={'data_path': 'output_data/train_data.nc', 'target_var': 'dbzh', 'architecture': 'ConvLSTM', 'seq_len': 10, 'pred_len': 1, 'epochs': 1, 'batch_size': 32, 'n_splits': 5, 'output_path': 'outputs/', 'exp_name': 'test_run_no_fill'}
X shape: (6071, 10, 6, 32, 32, 1)
y shape: (6071, 6, 32, 32, 1)


In [7]:
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    break

    seq_len, features = X.shape[1], X.shape[2]

    # Add callbacks if needed (e.g., EarlyStopping, ModelCheckpoint)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]

    if not os.path.exists(cfg.get("output_path")):
        os.makedirs(cfg.get("output_path"))

    print("Start Training!")
    model = build_model(params)

    model.optimizer.learning_rate = 1e-3

    # Train
    model = custom_fit(model, X_train, y_train, 1)
    # hist = model.fit(
    #     X_train, y_train,
    #     validation_data=(X_val, y_val),
    #     epochs=params["epochs"],
    #     batch_size=params["batch_size"],
    #     callbacks=callbacks
    # )

    # model.save(os.path.join(cfg.get("output_path"), f"{cfg.get('architecture')}_{model_id}_{cfg.get('exp_name')}.h5"))
    # np.save(os.path.join(cfg.get("output_path"), f"{cfg.get('architecture')}_{model_id}_hist_{cfg.get('exp_name')}.npy"), hist.history)
    model_id += 1

    # Evaluate on validation fold
    model_metric = model.evaluate(X_val, y_val)
    print(model_metric)
    break

In [11]:
X_train.shape

(4856, 10, 6, 32, 32, 1)

In [15]:
for x_a, y_a in train_dataset:
  break

In [18]:
x_a.shape

TensorShape([10, 6, 32, 32, 1])

In [17]:
y_a.shape

TensorShape([6, 32, 32, 1])

In [25]:
model(X_train[0])

ValueError: Exception encountered when calling ConvLSTM3D.call().

[1mInput 0 of layer "conv_lstm_cell" is incompatible with the layer: expected shape=(None, 6, 32, 32, 1), found shape=(10, 32, 32, 1, 1)[0m

Arguments received by ConvLSTM3D.call():
  • sequences=tf.Tensor(shape=(10, 6, 32, 32, 1, 1), dtype=float32)
  • initial_state=None
  • mask=None
  • training=False

In [20]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM3D, BatchNormalization, Conv3D

def build_model(input_shape=(None, None, None, None, 1), filters=[16, 8]):
    time, depth, height, width, channels = input_shape

    model = Sequential([
        tf.keras.layers.Input(shape=input_shape),
        ConvLSTM3D(
            filters=filters[0],
            kernel_size=(3,3,3),
            activation="tanh",
            padding="same",
            return_sequences=True,
            dropout=0.2
        ),
        BatchNormalization(),
        ConvLSTM3D(
            filters=filters[1],
            kernel_size=(3,3,3),
            activation="tanh",
            padding="same",
            return_sequences=False,
            dropout=0.2
        ),
        BatchNormalization(),
        # Final prediction: one frame (depth, height, width, channels)
        Conv3D(
            filters=channels,
            kernel_size=(3,3,3),
            activation="linear",
            padding="same"
        )
    ])

    model.compile(optimizer="adam", loss="mse")

    return model

In [22]:
model = ConvLSTMArch(input_shape=X_train.shape[1:], filters=params["filters"])
model.optimizer.learning_rate = 1e-3
loss_fn = model.loss

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [23]:
X_train.shape

(4856, 10, 6, 32, 32, 1)

In [9]:
model = build_model(params)
model.optimizer.learning_rate = 1e-3
loss_fn = model.loss

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
  with tf.GradientTape() as tape:
      y_pred = model(x_batch_train, training=True)

      # Create nan mask
      nan_mask = tf.math.is_nan(y_batch_train)
      nan_mask = tf.squeeze(nan_mask) # Remove extra dimensions if any

      y_pred_masked = tf.boolean_mask(y_pred, nan_mask)
      y_true_masked = tf.boolean_mask(y_batch_train, nan_mask)

      loss = loss_fn(y_true_masked, y_pred_masked)

  grads = tape.gradient(loss, model.trainable_variables)
  model.optimizer.apply_gradients(zip(grads, model.trainable_variables))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Exception encountered when calling ConvLSTM3D.call().

[1mInput 0 of layer "conv_lstm_cell" is incompatible with the layer: expected shape=(None, 6, 32, 32, 1), found shape=(10, 32, 32, 1, 1)[0m

Arguments received by ConvLSTM3D.call():
  • sequences=tf.Tensor(shape=(10, 6, 32, 32, 1, 1), dtype=float32)
  • initial_state=None
  • mask=None
  • training=True